In [1]:
from ultralytics import YOLO
from roboflow import Roboflow
from pathlib import Path
import cv2

In [2]:

rf = Roboflow(api_key="KoEpvrqTSdXhuxRYoc8h")
project = rf.workspace("huyens-workspace-coh7b").project("object-detection-project-fioui")
version = project.version(3)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...


In [4]:
model = YOLO("yolo11m.pt")

results = model.train(
    data = "/Users/huyenbach/Desktop/Denison/WHOI/happel_mooring/object-detection-project-3/data.yaml",
    epochs=100, imgsz=640, device = 'mps', batch=4, cache=False
)

Ultralytics 8.4.60 🚀 Python-3.11.5 torch-2.12.0 MPS (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/huyenbach/Desktop/Denison/WHOI/happel_mooring/object-detection-project-3/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-15, nbs=64, nms=False, opset=None, optimize=False, optimi

RuntimeError: MPS backend out of memory (MPS allocated: 8.98 GiB, other allocations: 102.94 MiB, max allowed: 9.07 GiB). Tried to allocate 100.00 MiB on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
import os

os.makedirs('./tracked-frames', exist_ok=True)

In [ ]:
model = YOLO("runs/detect/train-13/weights/best.pt")
video_path = Path("test_converted_videos/10.0.16.2_202309020548.mp4")
output_folder = "tracked-frames"

best_frames = {}
results = model.track(source=video_path, device='mps', stream=True, persist=True, tracker='botsort.yaml', conf=0.3)

for frame_index, result in enumerate(results):
    if result.boxes is None or result.boxes.id is None:
        continue

    raw_frame = result.orig_img
    boxes = result.boxes.xyxy.cpu().numpy()
    tracking_ids = result.boxes.id.cpu().numpy().astype(int)
    confidences = result.boxes.conf.cpu().numpy()

    for box, track_id, conf in zip(boxes, tracking_ids, confidences):
        x1, y1, x2, y2 = map(int, box)
        if track_id not in best_frames:
            best_frames[track_id] = {
                'highest_conf': conf,
                'frame_matrix': raw_frame.copy(),
                'box_coords': (x1,y1,x2,y2)
            }       

        else:
            if conf > best_frames[track_id]['highest_conf']:
                best_frames[track_id]['highest_conf'] = conf
                best_frames[track_id]['frame_matrix'] = raw_frame.copy()
                best_frames[track_id]['box_coords'] = (x1,y1,x2,y2)

for track_id, data in best_frames.items():
    full_frame = data['frame_matrix']
    x1,y1,x2,y2 = data['box_coords']
    conf_score = data['highest_conf']

    box_color = (0,255,0)
    label_text = f"ID: {track_id} ({conf_score:.2f})"
    
    # Draw the rectangle on the full image copy
    cv2.rectangle(full_frame, (x1, y1), (x2, y2), box_color, thickness=3)
    
    # Add the text label right above the box
    cv2.putText(
        full_frame, 
        label_text, 
        (x1, max(y1 - 10, 20)), 
        fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
        fontScale=0.7, 
        color=box_color, 
        thickness=2,
        lineType=cv2.LINE_AA)
    
    filename = f"organism_id_{track_id}_full_frame.jpg"
    save_path = os.path.join(output_folder, filename)
    cv2.imwrite(save_path, full_frame)


print(f"Done! Successfully generated {len(best_frames)} unique organism images.")